# Shared Setup

Shared constants and helper functions for the MobileNetV3-small notebook suite.

In [4]:
from __future__ import annotations

from meatlens_pork_pipeline.notebook_progress import (
    advance_notebook_cell_progress,
    finish_notebook_cell_progress,
    iter_notebook_progress,
    start_notebook_cell_progress,
)
NB_00_SHARED_SETUP_CELL_PROGRESS_1 = start_notebook_cell_progress('00_shared_setup.ipynb', 'Shared setup bootstrap', total_steps=1)

import os
import random
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from meatlens_pork_pipeline.windows_tf_bootstrap import bootstrap_windows_tensorflow_dll_paths

os.environ['PATH'] = bootstrap_windows_tensorflow_dll_paths(
    python_executable=Path(sys.executable),
    platform=sys.platform,
)

tf = None


def get_tensorflow() -> object:
    global tf
    if tf is None:
        try:
            import tensorflow as tensorflow_module
        except Exception as exc:
            raise RuntimeError('TensorFlow is unavailable, so GPU-backed training cannot start.') from exc

        tf = tensorflow_module
    return tf

NOTEBOOK_OVERRIDES = globals().get('NOTEBOOK_OVERRIDES', {})


def override(name: str, default: object) -> object:
    return NOTEBOOK_OVERRIDES.get(name, default)


NOTEBOOK_TEST_MODE = bool(override('NOTEBOOK_TEST_MODE', False))
DATA_ROOT = Path(str(override('DATA_ROOT', Path.cwd() / 'data')))
ROOT = Path(str(override('ROOT', Path.cwd())))
DATASET_SOURCE = str(override('DATASET_SOURCE', 'roboflow')).strip().lower()
if DATASET_SOURCE not in {'current', 'roboflow'}:
    raise ValueError("DATASET_SOURCE must be either 'current' or 'roboflow'.")
ROBOFLOW_DATASET_ROOT = Path(str(override('ROBOFLOW_DATASET_ROOT', ROOT / 'roboflow dataset')))
ROBOFLOW_PROCESSED_ROOT = Path(str(override('ROBOFLOW_PROCESSED_ROOT', DATA_ROOT / 'roboflow_processed_hsv_lab_threshold_roi_224')))
RAW_DATA_ROOT = Path(str(override('RAW_DATA_ROOT', DATA_ROOT / 'raw_dataset')))
default_generated_splits_root = ROOT / 'generated_splits' / 'roboflow' if DATASET_SOURCE == 'roboflow' else ROOT / 'generated_splits'
GENERATED_SPLITS_ROOT = Path(str(override('GENERATED_SPLITS_ROOT', default_generated_splits_root)))
default_processed_roi_root = ROBOFLOW_PROCESSED_ROOT if DATASET_SOURCE == 'roboflow' else DATA_ROOT / 'processed_hsv_lab_threshold_roi_224'
PROCESSED_ROI_ROOT = Path(str(override('PROCESSED_ROI_ROOT', default_processed_roi_root)))
RAW_CENTER_CROP_ROOT = Path(str(override('RAW_CENTER_CROP_ROOT', DATA_ROOT / 'raw_center_crop_224')))
CANONICAL_PROCESSING_SUMMARY_PATH = Path(
    str(override('CANONICAL_PROCESSING_SUMMARY_PATH', PROCESSED_ROI_ROOT / 'processing_summary.csv'))
)
default_training_outputs_root = ROOT / 'training_outputs' / 'roboflow' if DATASET_SOURCE == 'roboflow' else ROOT / 'training_outputs'
TRAINING_OUTPUTS_ROOT = Path(str(override('TRAINING_OUTPUTS_ROOT', default_training_outputs_root)))

INPUT_MODE = str(override('INPUT_MODE', 'processed_hsv_lab_threshold_roi_224'))
INPUT_MODE_ROOTS = {
    'processed_hsv_lab_threshold_roi_224': PROCESSED_ROI_ROOT,
    'raw_center_crop_224': RAW_CENTER_CROP_ROOT,
}
AUGMENTATION_PRESET = str(override('AUGMENTATION_PRESET', 'geometry_only_v1'))
SEVERE_ERROR_LABEL_PAIRS = [('fresh', 'spoiled'), ('spoiled', 'fresh')]

LABEL_ORDER = ['fresh', 'not fresh', 'spoiled']
RUN_SEEDS = [42, 123, 2026]
TARGET_SIZE = (224, 224)
INPUT_SHAPE = (224, 224, 3)
BATCH_SIZE = int(override('BATCH_SIZE', 32))
EPOCHS_HEAD = int(override('EPOCHS_HEAD', 8))
EPOCHS_FINE = int(override('EPOCHS_FINE', 20))
HEAD_LR = float(override('HEAD_LR', 1e-4))
FINE_TUNE_LR = float(override('FINE_TUNE_LR', 1e-5))
FINE_TUNE_FRACTION = float(override('FINE_TUNE_FRACTION', 0.25))
TRAINING_STRATEGY = str(override('TRAINING_STRATEGY', 'cached_embeddings_sgd_v1'))
REQUIRED_TRAINING_GPU_SUBSTRING = 'RTX 4050'


def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    if tf is not None:
        tf.keras.utils.set_random_seed(seed)
        try:
            tf.config.experimental.enable_op_determinism()
        except Exception:
            pass


def inspect_default_processed_dataset(
    processed_root: Path = PROCESSED_ROI_ROOT,
    summary_path: Path = CANONICAL_PROCESSING_SUMMARY_PATH,
) -> dict[str, object]:
    sample_dirs = sorted(path.name for path in processed_root.glob('sample *') if path.is_dir())
    summary: dict[str, object] = {
        'processed_root': str(processed_root),
        'processing_summary_path': str(summary_path),
        'sample_dirs': sample_dirs,
        'sample_dir_count': len(sample_dirs),
        'manifest_exists': summary_path.exists(),
        'manifest_rows': 0,
        'label_counts': {},
        'ready': bool(sample_dirs) and summary_path.exists(),
    }

    if not summary_path.exists():
        return summary

    manifest_df = pd.read_csv(summary_path, dtype=str).fillna('')
    label_counts: dict[str, int] = {}
    if 'label' in manifest_df.columns:
        label_counts = (
            manifest_df['label']
            .astype(str)
            .str.strip()
            .str.lower()
            .value_counts()
            .to_dict()
        )

    summary['manifest_rows'] = int(len(manifest_df))
    summary['label_counts'] = label_counts
    summary['ready'] = bool(sample_dirs) and int(len(manifest_df)) > 0
    return summary


def resolve_input_root(input_mode: str) -> Path:
    normalized = str(input_mode).strip()
    if normalized not in INPUT_MODE_ROOTS:
        raise ValueError(
            f'Unsupported input_mode {normalized!r}. Expected one of {sorted(INPUT_MODE_ROOTS)}'
        )
    return INPUT_MODE_ROOTS[normalized]


def resolve_fine_tune_fraction(value: object) -> float:
    fraction = float(value)
    allowed = {0.0, 0.25, 1.0}
    if fraction not in allowed:
        raise ValueError(f'Fine-tune fraction must be one of {sorted(allowed)}. Got {fraction}.')
    return fraction


def build_sample_heldout_validation_split(
    processed_df: pd.DataFrame,
    val_size: float,
    random_state: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    if 'sample_id' not in processed_df.columns:
        raise ValueError('Processed manifest must contain sample_id for sample-held-out validation.')

    grouped = (
        processed_df[['sample_id', 'label']]
        .drop_duplicates()
        .groupby('sample_id')['label']
        .agg(lambda values: tuple(sorted(values)))
        .reset_index()
    )
    if grouped.empty:
        raise ValueError('Cannot build a validation split from an empty processed manifest.')

    candidate_count = max(1, int(round(len(grouped) * float(val_size))))
    shuffled = grouped.sample(frac=1.0, random_state=random_state).reset_index(drop=True)
    selected_sample_ids = shuffled.head(candidate_count)['sample_id'].astype(str).tolist()

    train_df = processed_df[~processed_df['sample_id'].astype(str).isin(selected_sample_ids)].copy()
    val_df = processed_df[processed_df['sample_id'].astype(str).isin(selected_sample_ids)].copy()

    if train_df.empty or val_df.empty:
        raise ValueError('Sample-held-out deployment split produced an empty train or validation partition.')

    return train_df.reset_index(drop=True), val_df.reset_index(drop=True)


DEFAULT_PROCESSED_DATASET_SUMMARY = inspect_default_processed_dataset()
DEFAULT_PROCESSED_DATASET_READY = bool(DEFAULT_PROCESSED_DATASET_SUMMARY['ready'])


def enforce_training_gpu(required_name_substring: str = REQUIRED_TRAINING_GPU_SUBSTRING) -> list[str]:
    if NOTEBOOK_TEST_MODE or bool(override('SKIP_GPU_CHECK', False)):
        return []

    try:
        result = subprocess.run(
            ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
            check=True,
            capture_output=True,
            text=True,
        )
    except Exception as exc:
        raise RuntimeError('Unable to verify NVIDIA GPU availability via nvidia-smi.') from exc

    gpu_names = [line.strip() for line in result.stdout.splitlines() if line.strip()]
    if not gpu_names:
        raise RuntimeError('No NVIDIA GPU detected. Training must run on the laptop RTX 4050.')

    tensorflow_module = get_tensorflow()
    tensorflow_gpus = tensorflow_module.config.list_physical_devices('GPU')
    if not tensorflow_gpus:
        raise RuntimeError('TensorFlow does not currently see any GPU devices. Fix CUDA/TensorFlow GPU support before training.')

    normalized_required = required_name_substring.lower()
    if not any(normalized_required in gpu_name.lower() for gpu_name in gpu_names):
        raise RuntimeError(
            f'Required GPU containing {required_name_substring!r} was not found. Detected GPUs: {gpu_names}'
        )
    return gpu_names


for path in (GENERATED_SPLITS_ROOT, TRAINING_OUTPUTS_ROOT):
    ensure_dir(path)

if DEFAULT_PROCESSED_DATASET_READY:
    print(
        'Default processed dataset ready:',
        DEFAULT_PROCESSED_DATASET_SUMMARY['manifest_rows'],
        'images across',
        DEFAULT_PROCESSED_DATASET_SUMMARY['sample_dir_count'],
        'sample folders.',
    )
else:
    print(
        'Default processed dataset not ready at',
        PROCESSED_ROI_ROOT,
        '- run the early notebooks with raw or Excel input, or set overrides.',
    )

finish_notebook_cell_progress(NB_00_SHARED_SETUP_CELL_PROGRESS_1)


[START] 00_shared_setup.ipynb | Shared setup bootstrap [0/1 step] elapsed=0.0s
Default processed dataset not ready at C:\Users\Adriaan M. Dimate\Desktop\development\school\meatlens-training-2\data\roboflow_processed_hsv_lab_threshold_roi_224 - run the early notebooks with raw or Excel input, or set overrides.
[RUNNING] 00_shared_setup.ipynb | Shared setup bootstrap | done [0/1 step] elapsed=0.0s
[RUNNING] 00_shared_setup.ipynb | Shared setup bootstrap | done [1/1 step] elapsed=0.0s
